# Session 4: when price and demand move together
**A. Real-world problem.** In Session 3 the bakery's price, variable cost and demand were
random, but *independent*: a high price said nothing about demand. Real customers do not
behave like that. When the price is higher than planned, fewer people buy; in a festival week
a higher price can even come with more buyers. How much does that link change the risk of a
loss month, even when every average stays the same? Today you add a **correlation** between
price and demand to the Monte Carlo model, check it against formulas, and ask what correlation
a lender should believe.

*General education only: the bakery and its numbers are made up, and this is not
financial advice.*

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

F, P, V, Q = 1200.0, 3.50, 1.40, 800.0          # as in Sessions 2 and 3
P_SD, V_SD, Q_SD = 0.20, 0.15, 150.0            # made-up standard deviations of Session 3
DEMAND_B = 250.0                                # Session 2: loaves per month lost per dollar of price

## Step 1: correlated random draws
The **correlation** rho (a number from -1 to +1) says how two random inputs move together:
-1 means a high price always comes with low demand, 0 means no link, +1 means they rise
together. To draw demand with a chosen rho we mix two independent standard normal numbers:
`z_demand = rho * z_price + sqrt(1 - rho^2) * z_other`. Then the demand is `mean + sd * z_demand`.
The price and the variable cost are drawn exactly as in Session 3, so with rho = 0 this
reproduces Session 3 to the last digit.

In [ ]:
def simulate_profit_corr(rho=0.0, n=20000, seed=42, p_mu=P, p_sd=P_SD, v_mu=V, v_sd=V_SD,
                         d_mu=Q, d_sd=Q_SD, fixed=F, inputs=False):
    """Monthly profit of n random months; correlation rho between price and demand (variable cost independent).

    With inputs=True also return the price and demand arrays."""
    if not -1.0 <= rho <= 1.0:
        raise ValueError("the correlation must be between -1 and +1")
    rng = np.random.default_rng(seed)
    z_price = rng.standard_normal(n)
    z_cost = rng.standard_normal(n)
    z_other = rng.standard_normal(n)
    price = p_mu + p_sd * z_price
    vcost = v_mu + v_sd * z_cost
    demand = np.maximum(d_mu + d_sd * (rho * z_price + math.sqrt(1.0 - rho * rho) * z_other), 0.0)
    profit = (price - vcost) * demand - fixed
    return (profit, price, demand) if inputs else profit


def summarize(profits):
    return {"mean": float(np.mean(profits)), "std": float(np.std(profits)),
            "p5": float(np.percentile(profits, 5)), "p50": float(np.percentile(profits, 50)),
            "p95": float(np.percentile(profits, 95)), "loss": float(np.mean(profits < 0))}


def loss_probability(rho, **kw):
    return float(np.mean(simulate_profit_corr(rho, **kw) < 0))

## Step 2: formulas to check the simulation against
Write profit = X * D - F with X = price - variable cost and D = demand. X and D are jointly
normal, so the mean and the standard deviation of X D have exact formulas (Isserlis' theorem):
* mean = mu_X mu_D + Cov(X, D) - F, and Cov(X, D) = rho sigma_p sigma_D (the cost is independent),
* variance = mu_X^2 sigma_D^2 + mu_D^2 sigma_X^2 + 2 mu_X mu_D Cov + sigma_X^2 sigma_D^2 + Cov^2,
  with sigma_X^2 = sigma_p^2 + sigma_v^2.
These ignore the rare months where demand would be negative (clipped at 0 in the simulation;
at 800 with sd 150 that is more than 5 standard deviations away).

In [ ]:
def profit_mean_formula(rho, p_mu=P, p_sd=P_SD, v_mu=V, d_mu=Q, d_sd=Q_SD, fixed=F):
    return (p_mu - v_mu) * d_mu + rho * p_sd * d_sd - fixed


def profit_std_formula(rho, p_mu=P, p_sd=P_SD, v_mu=V, v_sd=V_SD, d_mu=Q, d_sd=Q_SD):
    mu_x, var_x = p_mu - v_mu, p_sd ** 2 + v_sd ** 2
    cov = rho * p_sd * d_sd
    var = mu_x ** 2 * d_sd ** 2 + d_mu ** 2 * var_x + 2.0 * mu_x * d_mu * cov + var_x * d_sd ** 2 + cov ** 2
    return math.sqrt(var)


def loss_normal_approx(rho, **kw):
    """Quick estimate Phi(-mean / std). The profit is a product of normals, so it is skewed and this is only rough."""
    z = -profit_mean_formula(rho, **{k: v for k, v in kw.items() if k in ("p_mu", "p_sd", "v_mu", "d_mu", "d_sd", "fixed")}) / \
        profit_std_formula(rho, **{k: v for k, v in kw.items() if k in ("p_mu", "p_sd", "v_mu", "v_sd", "d_mu", "d_sd")})
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

## Step 3: what correlation is believable?
Session 2 gave a demand line: each extra dollar of price loses 250 loaves a month. If the
price wobbles by sigma_p = 0.20 dollars *and demand follows that line*, demand moves by
250 x 0.20 = 50 loaves, and since the whole demand spread is 150 loaves the correlation is
-50 / 150 = -0.33. That is a plausible central value; a stronger link needs more of the demand
spread to be explained by price.

In [ ]:
def implied_rho(p_sd=P_SD, d_sd=Q_SD, slope=DEMAND_B):
    """Correlation if demand moves along a demand line of the given slope when the price wobbles."""
    r = -slope * p_sd / d_sd
    if r < -1.0:
        raise ValueError("the demand line alone would explain more than the whole demand spread")
    return r

## Step 4: how the risk depends on the correlation
We scan rho from -0.9 to +0.9 with the same seed (common random numbers, so the curve is
smooth). The mean profit hardly moves (it shifts by rho sigma_p sigma_D = 30 dollars per unit
of rho) but the spread, and with it the loss probability, changes a lot. The lender's limit
turns that into a question: for which correlations is the loss probability at most the limit?

In [ ]:
def loss_vs_rho(rhos=None, **kw):
    rhos = np.round(np.arange(-0.9, 0.9001, 0.1), 10) if rhos is None else rhos
    return [(float(r) + 0.0, loss_probability(float(r), **kw)) for r in rhos]


def rho_for_limit(limit, lo=-0.99, hi=0.99, **kw):
    """Largest correlation whose loss probability is at most `limit` (bisection), or None if even `lo` fails."""
    if loss_probability(lo, **kw) > limit:
        return None
    if loss_probability(hi, **kw) <= limit:
        return hi
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if loss_probability(mid, **kw) <= limit:
            lo = mid
        else:
            hi = mid
    return lo

## Step 5: a note on what the correlation does
A high price earns more per loaf but sells fewer loaves. With rho < 0 the two effects partly
cancel (a **natural hedge**), so the profit varies less; with rho > 0 they add and the profit
varies more. The expected profit changes only a little. So: *the risk of a loss depends on the
links between the inputs, not only on the spread of each one.*

In [ ]:
def decompose_spread(rho):
    """Standard deviation of profit with rho, and with rho = 0, and their difference."""
    s, s0 = profit_std_formula(rho), profit_std_formula(0.0)
    return s, s0, s - s0

## See it
Figure 1: price against demand for 1500 months at rho = 0 and rho = -0.5, with the loss
months marked. Figure 2: profit histograms for rho = -0.5, 0 and +0.5, and the loss
probability against rho with a limit.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("rho: loss probability, mean, std (simulated), mean, std (formula), normal estimate")
    for rho, loss in loss_vs_rho():
        pr = simulate_profit_corr(rho)
        print(f"{rho:5.1f}: loss {100 * loss:5.2f} %  mean {pr.mean():6.1f} std {pr.std():6.1f}   "
              f"formula {profit_mean_formula(rho):6.1f} {profit_std_formula(rho):6.1f}   normal {100 * loss_normal_approx(rho):5.2f} %")
    print("rho implied by the demand line:", round(implied_rho(), 3), "loss", round(100 * loss_probability(implied_rho()), 2), "%")
    r8 = rho_for_limit(0.08)
    print("largest rho with a loss probability of at most 8 %:", round(r8, 3))

    BLUE, ORANGE, MAGENTA = "#0072b2", "#c86a00", "#d81b60"
    fig, axs = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)
    for ax, rho in zip(axs, (0.0, -0.5)):
        pr, price, dem = simulate_profit_corr(rho, n=1500, inputs=True)
        loss = pr < 0
        ax.scatter(price[~loss], dem[~loss], s=8, color=BLUE, alpha=0.55, label="profit month")
        ax.scatter(price[loss], dem[loss], s=10, color=MAGENTA, marker="x", label="loss month")
        ax.set_xlabel("price drawn ($ per loaf)")
        ax.set_title(f"correlation {rho:+.1f}: {100 * np.mean(loss):.1f} % of months lose money", fontsize=10)
        ax.grid(alpha=0.3)
    axs[0].set_ylabel("demand drawn (loaves per month)")
    axs[0].legend(fontsize=8, loc="lower left")
    fig.tight_layout()
    fig.savefig(OUT / "price_demand_scatter.png", dpi=120)

    fig2, axs2 = plt.subplots(1, 2, figsize=(9.6, 4.0))
    bins = np.linspace(-1000, 2500, 71)
    for rho, col, ls in ((-0.5, BLUE, "-"), (0.0, "#444444", "--"), (0.5, ORANGE, ":")):
        pr = simulate_profit_corr(rho)
        axs2[0].hist(pr, bins=bins, histtype="step", lw=2, color=col, ls=ls, label=f"correlation {rho:+.1f}")
    axs2[0].axvline(0, color="black", lw=0.8)
    axs2[0].set_xlabel("monthly profit ($)")
    axs2[0].set_ylabel("simulated months (of 20000)")
    axs2[0].set_title("Same averages, different spread", fontsize=10)
    axs2[0].legend(fontsize=8)
    axs2[0].grid(alpha=0.3)
    rs, ls_ = zip(*loss_vs_rho())
    axs2[1].plot(rs, [100 * x for x in ls_], color=BLUE, lw=2, marker="o", ms=4, label="simulated loss probability")
    axs2[1].axhline(8, color=MAGENTA, ls="--", lw=1.5, label="lender's limit 8 %")
    axs2[1].axvline(implied_rho(), color=ORANGE, ls=":", lw=2, label="implied by the demand line (-0.33)")
    axs2[1].set_xlabel("correlation between price and demand")
    axs2[1].set_ylabel("chance a month makes a loss (%)")
    axs2[1].set_title("Risk grows with the correlation", fontsize=10)
    axs2[1].legend(fontsize=8)
    axs2[1].grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "loss_vs_rho.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Run `loss_probability(rho)` for rho = -0.8, -0.4, 0, 0.4 and 0.8 and write down how much the loss probability changes for each step of 0.4. Is the change the same size everywhere?
2. **Stretch.** Make the price wobble larger (`p_sd=0.40`) and find, with `implied_rho(p_sd=0.40)` and `loss_probability`, how the loss probability at the demand-line correlation changes. Explain why `implied_rho` raises an error when the demand line explains more than the whole demand spread.
3. **Explain.** A friend says: "if price and demand are negatively correlated the bakery is hurt, because the correlation is negative". Explain with the picture of the price-demand cloud why that is wrong for the risk of a loss month, and when a negative correlation would not help.

## Assessment
- **A. Conceptual.** Why does a negative correlation between price and demand make the bakery's profit less risky, although it hardly changes the average profit?
- **B. Numerical.** Find the mean and the standard deviation of monthly profit for correlation +0.5 by hand (same data as the worked example), and use them to estimate the loss probability with the normal shortcut. How does it compare with the 12.4 percent of the simulation?
- **C. Graph and simulation interpretation.** In the loss-probability figure, the independent case (correlation 0) lies at 9.5 percent, above the lender's 8 percent, and the demand-line value (-0.33) lies below it. What can you conclude, and what can you not conclude?
- **D. Design and optimisation challenge.** Design: a lender asks for a loss probability of at most 5 percent. The owner can cut the fixed cost in steps of 50 dollars from 1200. (a) Assuming independent inputs, what is the highest fixed cost that meets the limit? (b) Assuming the demand-line correlation of -0.33? How much does believing the correlation relax the budget, and how safe is that answer?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    indep = L.loss_probability(0.0)
    # --- reproduces Session 3
    s0 = L.summarize(L.simulate_profit_corr(0.0))
    checks.append(("rho = 0 reproduces Session 3: loss 9.5 percent, mean $476.4, standard deviation $374.9",
                   abs(s0["loss"] - 0.095) < 1e-9 and abs(s0["mean"] - 476.4) < 0.1 and abs(s0["std"] - 374.9) < 0.1))
    checks.append(("seeded run: the same numbers every time", L.summarize(L.simulate_profit_corr(-0.5)) == L.summarize(L.simulate_profit_corr(-0.5))))
    # --- roadmap checkpoint
    neg = L.loss_probability(-0.5)
    checks.append((f"roadmap check: with correlation -0.5 the loss probability is {100 * neg:.1f} percent against {100 * indep:.1f} percent independent, at least 2 points lower",
                   indep - neg >= 0.02 and abs(neg - 0.0559) < 0.001))
    ok_seeds = all(L.loss_probability(0.0, seed=s) - L.loss_probability(-0.5, seed=s) >= 0.02 for s in range(1, 8))
    checks.append(("the 2-point difference is not a lucky seed: it holds for seeds 1 to 7", ok_seeds))
    pos = L.loss_probability(0.5)
    checks.append((f"positive correlation raises the risk: +0.5 gives {100 * pos:.1f} percent, more than the independent {100 * indep:.1f}",
                   pos - indep >= 0.02 and abs(pos - 0.124) < 0.001))
    ls = [x for _, x in L.loss_vs_rho()]
    checks.append(("the loss probability rises step by step with the correlation from -0.9 (2.6 percent) to +0.9 (14.4 percent)",
                   all(b > a for a, b in zip(ls, ls[1:])) and abs(ls[0] - 0.0263) < 0.001 and abs(ls[-1] - 0.1436) < 0.001))
    # --- the draws really have the requested correlation
    for rho in (-0.7, 0.0, 0.4):
        _, price, dem = L.simulate_profit_corr(rho, n=100000, inputs=True)
        got = float(np.corrcoef(price, dem)[0, 1])
        checks.append((f"validation: the sample correlation of price and demand is {got:+.3f} for rho = {rho:+.1f}", abs(got - rho) < 0.01))
    _, price, dem = L.simulate_profit_corr(-0.5, n=100000, inputs=True)
    checks.append(("each input keeps its own mean and spread: price 3.50 and 0.20, demand 800 and 150",
                   abs(price.mean() - 3.5) < 0.01 and abs(price.std() - 0.2) < 0.005 and abs(dem.mean() - 800) < 2.0 and abs(dem.std() - 150) < 2.0))
    # --- formulas
    worst_m, worst_s = 0.0, 0.0
    for rho in (-0.9, -0.5, 0.0, 0.5, 0.9):
        pr = L.simulate_profit_corr(rho, n=200000)
        worst_m = max(worst_m, abs(pr.mean() - L.profit_mean_formula(rho)))
        worst_s = max(worst_s, abs(pr.std() / L.profit_std_formula(rho) - 1.0))
    checks.append((f"validation: simulated mean profit is within $3 of the formula (worst ${worst_m:.2f})", worst_m < 3.0))
    checks.append((f"validation: simulated standard deviation is within 1 percent of the exact formula (worst {100 * worst_s:.2f} percent)", worst_s < 0.01))
    checks.append(("worked example: rho = -0.5 gives mean $465.0 and standard deviation $300.8 by hand",
                   abs(L.profit_mean_formula(-0.5) - 465.0) < 1e-9 and abs(L.profit_std_formula(-0.5) - 300.8) < 0.05))
    checks.append(("the correlation moves the mean by only rho x 0.20 x 150 = 30 dollars per unit of rho",
                   abs((L.profit_mean_formula(1.0) - L.profit_mean_formula(0.0)) - 30.0) < 1e-9))
    s, s_0, diff = L.decompose_spread(-0.5)
    checks.append((f"a negative correlation shrinks the spread: ${s:.1f} against ${s_0:.1f} ({diff:.1f})", s < s_0 and abs(diff + 74.2) < 0.2))
    approx = L.loss_normal_approx(-0.5)
    checks.append((f"the normal shortcut Phi(-mean/std) gives {100 * approx:.1f} percent for rho = -0.5, within 1 point of the simulation but a little high (skewed profit)",
                   abs(approx - neg) < 0.01 and approx > neg))
    # --- implied correlation and the lender's limit
    ir = L.implied_rho()
    checks.append((f"the demand line of Session 2 implies a correlation of {ir:.3f} (-250 x 0.20 / 150)", abs(ir + 1.0 / 3.0) < 1e-9))
    checks.append(("limit: a demand line steeper than the whole demand spread is rejected", _raises(lambda: L.implied_rho(d_sd=40.0))))
    li = L.loss_probability(ir)
    checks.append((f"at the implied correlation the loss probability is {100 * li:.1f} percent, between the independent and the -0.5 value", neg < li < indep))
    r8 = L.rho_for_limit(0.08)
    checks.append((f"mini-project: the largest correlation with a loss probability of at most 8 percent is {r8:.3f}",
                   r8 is not None and abs(r8 + 0.189) < 0.01 and L.loss_probability(r8) <= 0.08 < L.loss_probability(r8 + 0.01)))
    checks.append(("mini-project: an 8 percent limit holds at the implied correlation and fails when the inputs are independent",
                   L.loss_probability(ir) <= 0.08 and indep > 0.08))
    checks.append(("edge: no correlation can reach a 1 percent limit, and every correlation meets a 20 percent limit",
                   L.rho_for_limit(0.01) is None and L.rho_for_limit(0.20) == 0.99))
    # --- limits and edges
    checks.append(("limit: a correlation outside -1 to +1 is rejected", _raises(lambda: L.simulate_profit_corr(1.2)) and _raises(lambda: L.simulate_profit_corr(-1.01))))
    pr1 = L.simulate_profit_corr(-1.0, n=50000, inputs=True)
    checks.append(("edge: correlation -1 puts every draw exactly on one line (demand falls 750 loaves per dollar of price)",
                   abs(float(np.corrcoef(pr1[1], pr1[2])[0, 1]) + 1.0) < 1e-9 and abs(float(np.polyfit(pr1[1], pr1[2], 1)[0]) + 750.0) < 1.0))
    checks.append(("limit: with no randomness at all the profit is exactly $480 every month",
                   np.allclose(L.simulate_profit_corr(0.5, n=50, p_sd=0, v_sd=0, d_sd=0), 480.0)))
    checks.append(("a price that matches the variable cost with certain demand gives a loss of exactly the fixed cost",
                   np.allclose(L.simulate_profit_corr(0.0, n=20, p_mu=1.4, p_sd=0, v_sd=0, d_sd=0), -1200.0)))
    return checks


def _raises(fn):
    try:
        fn()
    except ValueError:
        return True
    return False

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
